# Numerai 13F-HR Data Updater (Horizontal Column Version)

このノートブックは、SEC EDGAR APIから最新の13F-HRデータを取得し、Excelファイル内の**シート「ALL」**に対して、**新しい日付の列**を追加します。

### 動作仕様:
1. `CUSIP` を主キーとして既存データを照合します。
2. 最新の報告日を新しい列として右端に追加します。
3. 既存の銘柄には金額を上書き/追記し、新規銘柄は新しい行として追加します。

In [ ]:
# 必要なライブラリのインストール
!pip install requests pandas beautifulsoup4 openpyxl lxml -q

import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
from google.colab import drive

# --- 設定 ---
CIK = "0001668527"
USER_AGENT = "Kei Sanada sdk7777@gmail.com"
SEC_HEADERS = {"User-Agent": USER_AGENT}

# Googleドライブをマウント
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Excelファイルのパスとシート名
EXCEL_PATH = "/content/drive/MyDrive/History_Numerai_13F-HR.xlsx"
SHEET_NAME = "ALL"

### 1. SECから最新データを取得・パースする

In [ ]:
def get_latest_data(cik):
    # 1. 提出物リスト取得
    sub_url = f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json"
    r = requests.get(sub_url, headers=SEC_HEADERS)
    r.raise_for_status()
    recent = r.json()['filings']['recent']
    
    acc_num, report_date = None, None
    for i, form in enumerate(recent['form']):
        if form == '13F-HR':
            acc_num = recent['accessionNumber'][i].replace('-', '')
            report_date = recent['reportDate'][i]
            break
    
    if not acc_num: return None, None

    # 2. XML特定
    idx_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/index.json"
    idx_r = requests.get(idx_url, headers=SEC_HEADERS)
    idx_r.raise_for_status()
    
    xml_file = None
    for f in idx_r.json()['directory']['item']:
        if 'informationtable' in f['name'].lower() and f['name'].endswith('.xml'):
            xml_file = f['name']
            break
    
    if not xml_file:
        for f in idx_r.json()['directory']['item']:
            if f['name'].endswith('.xml') and 'primary' not in f['name'].lower():
                xml_file = f['name']
                break

    # 3. XMLパース
    xml_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/{xml_file}"
    print(f"最新の報告日 {report_date} のデータを取得中...")
    xml_r = requests.get(xml_url, headers=SEC_HEADERS)
    xml_r.raise_for_status()
    
    soup = BeautifulSoup(xml_r.content, 'xml')
    results = []
    for info in soup.find_all(['infoTable', 'ns1:infoTable']):
        results.append({
            'CUSIP': info.find(['cusip', 'ns1:cusip']).text,
            'ISSUER NAME': info.find(['nameOfIssuer', 'ns1:nameOfIssuer']).text,
            'Value': int(info.find(['value', 'ns1:value']).text or 0)
        })
    
    # 同一CUSIPが複数ある場合は合計する（オプション種別等がある場合のため）
    df_new = pd.DataFrame(results).groupby(['CUSIP', 'ISSUER NAME'])['Value'].sum().reset_index()
    return df_new, report_date

df_latest, latest_date = get_latest_data(CIK)

### 2. Excelのシート「ALL」を横方向に更新する

In [ ]:
if df_latest is not None:
    if os.path.exists(EXCEL_PATH):
        # 既存の全シートを読み込み（保存時に他のシートを消さないため）
        excel_file = pd.ExcelFile(EXCEL_PATH)
        all_sheets = {name: excel_file.parse(name) for name in excel_file.sheet_names}
        
        if SHEET_NAME in all_sheets:
            df_all = all_sheets[SHEET_NAME]
            
            # Rename columns in df_all to match df_latest for merging
            df_all = df_all.rename(columns={'cusip': 'CUSIP', 'nameOfIssuer': 'ISSUER NAME'})

            # 既にその日付の列があるか確認
            if latest_date in df_all.columns:
                print(f"警告: 既に列 '{latest_date}' が存在します。上書きします。")
            
            # CUSIPをキーにしてマージ（既存の行は維持、新しい銘柄は追加）
            # how='outer' で新規銘柄（最新データにあって既存にないもの）を行として追加
            df_all = pd.merge(
                df_all, 
                df_latest.rename(columns={'Value': latest_date}), 
                on=['CUSIP', 'ISSUER NAME'], 
                how='outer'
            )
            
            all_sheets[SHEET_NAME] = df_all
            
            # Excelに書き戻し
            with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
                for name, df_sheet in all_sheets.items():
                    df_sheet.to_excel(writer, sheet_name=name, index=False)
            
            print(f"成功: シート '{SHEET_NAME}' に列 '{latest_date}' を追加しました。")
        else:
            print(f"エラー: シート '{SHEET_NAME}' が見つかりません。")
    else:
        # 新規作成の場合
        df_latest.rename(columns={'Value': latest_date}).to_excel(EXCEL_PATH, sheet_name=SHEET_NAME, index=False)
        print(f"新規ファイルとして作成しました。")
else:
    print("データの取得に失敗しました。")